# 37 — Is the production posterior right *for this window*? L-C2ST (System I, N3)

SBC and TARP test the posterior on average over the prior; the coverage tables test it per scenario. They cannot say whether the posterior at one particular observation is correct. The **local classifier two-sample test (L-C2ST**, Linhart et al. 2023;
`sbi.diagnostics.LC2ST`) can: a classifier is trained to tell apart (θ from the prior, x) from (θ from the posterior, x) on a calibration set; at a given x_o, the classifier's accuracy on posterior draws θ_o ~ q(θ | x_o) is compared with its distribution under the
null (a posterior that is exactly right). A small p-value says that the posterior at x_o differs from the truth in a way the classifier can see. This replaces the NUTS ground truth (decision 2026-11-09; the diagnosis is in the plan, SI-1).

**Settings (fixed before running; runbook §2f row e2).** sbi 0.26.1 `LC2ST` with its **defaults**: sklearn MLPClassifier with hidden layers (10·dim θ, 10·dim θ) = (20, 20), adam, early stopping, 1 fold, 1 classifier, seed 1, 100 null trials, permutation null, no z-scoring.
**Calibration set:** N_cal = 20,000 pairs (θ from the prior; x simulated under the production model: protocol P2, N3 sensors, true inlet from the notebook-35 distribution, *measured* Ti, Tci with ε = 0.1 K; new seeds 9 500 001–9 500 004) with one draw from the production ensemble (notebook 36) at each x.
**Test windows:** the first 5 windows of each scenario Sc1–Sc5 in D1 (25 windows, fixed by index), 2,000 ensemble draws per window as θ_o, x_o = the window's S1m features. **Reported:** the per-window p-value, the number below 0.05 (about 1.25 expected under a calibrated posterior) and the minimum; no decision rule.
**Power check (positive control, same classifier and null):** the p-values of deliberately wrong posteriors for the same windows (β shifted by +0.05 and +0.10; the width halved). If the default classifier has no power the commentary says so and a larger classifier is reported as a labelled sensitivity.


In [1]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent
for p in (ROOT / "src", ROOT / "scripts"):
    sys.path.insert(0, str(p))
os.environ["CSTR_SENSOR_MODEL"] = "N3"
warnings.filterwarnings("ignore")
import importlib, ws_eval, ws_phase3, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet
for _m in (ws_eval, ws_phase3, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet):
    importlib.reload(_m)
we, P3, M, P36, wi = ws_eval, ws_phase3, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet
from sbi.diagnostics import LC2ST
R = ROOT / "results"; OUTD = R / "37_N3"; OUTD.mkdir(exist_ok=True); T0 = time.time()
N_CAL, N_THETA_O, N_NULL = 20_000, 2_000, 100
post, prod = P3.load_production_posterior()
print("Production posterior:", prod["production_budget"], "simulations, ensemble of", len(prod["members"]))


Production posterior: 20000 simulations, ensemble of 5


## 1. Calibration set under the production model (cached)

In [2]:
f = OUTD / "cal.npz"
if f.exists():
    c = we.load_npz(f); th, X = c["theta"], c["x"]
else:
    th = we.prior_draws(N_CAL, 9_500_001); inl = M._inlets(N_CAL, 9_500_002)
    b29 = wi.simulate(th, inl, M.PROTOCOL, 9_500_003, features="S0_base29", **M.NOISE)
    X = P36.features(b29, inl, 9_500_004); we.save_npz(f, {"theta": th, "x": X, "inlet": inl})
assert th.shape == (N_CAL, 2) and X.shape == (N_CAL, 34) and np.isfinite(X).all()
smp, leak = we.sample_within_prior(post, X, 1, rng_seed=1)
print(f"calibration set {X.shape}; one ensemble draw per x (prior leakage {leak.mean():.4f}) | {time.time() - T0:.0f}s")
T_ = torch.tensor(th, dtype=torch.float32); X_ = torch.tensor(X, dtype=torch.float32); S_ = torch.tensor(smp[:, 0, :], dtype=torch.float32)


calibration set (20000, 34); one ensemble draw per x (prior leakage 0.0068) | 90s


## 2. Train the L-C2ST classifier and its null (sbi defaults)

In [3]:
t = time.time(); lc = LC2ST(T_, X_, S_, seed=1, num_trials_null=N_NULL)
lc.train_on_observed_data(); lc.train_under_null_hypothesis(verbosity=0)
print(f"classifier and {N_NULL} null classifiers trained in {time.time() - t:.0f}s")


classifier and 100 null classifiers trained in 307s


## 3. The 25 test windows: per-window p-values

In [4]:
d1 = P3.load_d1(); Xo_all = P3.production_features(d1); sid = d1["scenario_id"]
SCN = {1: "Sc1", 2: "Sc2", 3: "Sc3", 4: "Sc4", 5: "Sc5"}
wins = np.concatenate([np.where(sid == s)[0][:5] for s in SCN])
so, _ = we.sample_within_prior(post, Xo_all[wins], N_THETA_O, rng_seed=2)
def pvals(samples):
    return np.array([lc.p_value(torch.tensor(samples[k], dtype=torch.float32), torch.tensor(Xo_all[w], dtype=torch.float32)) for k, w in enumerate(wins)])
p_main = pvals(so)
res = pd.DataFrame(dict(window=wins, scenario=[SCN[int(sid[w])] for w in wins], alpha_true=d1["theta"][wins, 2], beta_true=d1["theta"][wins, 3],
                        post_mean_alpha=so.mean(1)[:, 0], post_mean_beta=so.mean(1)[:, 1], p_value=p_main))
display(res.round(3)); res.to_csv(R / "37_N3_lc2st_pvalues.csv", index=False)
from scipy import stats
print(f"p < 0.05 in {(p_main < 0.05).sum()} of {len(p_main)} windows (expected about {0.05 * len(p_main):.2f} under a calibrated posterior) | minimum p = {p_main.min():.3f} | "
      f"median p = {np.median(p_main):.3f} | KS test of the 25 p-values against U(0,1): p = {stats.kstest(p_main, 'uniform').pvalue:.3f}")
print("the null is estimated with 100 trials: the smallest attainable p-value is about 0.01")


,window,scenario,alpha_true,beta_true,post_mean_alpha,post_mean_beta,p_value
0,0,Sc1,1.00,1.00,0.958,0.920,0.08
1,1,Sc1,1.00,1.00,0.986,1.013,0.00
2,2,Sc1,1.00,1.00,1.020,1.028,0.02
3,3,Sc1,1.00,1.00,0.993,1.012,0.17
4,4,Sc1,1.00,1.00,0.982,1.035,0.57
5,50,Sc2,1.00,0.85,0.962,0.891,0.00
6,51,Sc2,1.00,0.85,1.041,0.850,0.00
7,52,Sc2,1.00,0.85,1.006,0.820,0.53
8,53,Sc2,1.00,0.85,1.035,0.835,0.00
9,54,Sc2,1.00,0.85,0.956,0.841,0.00


p < 0.05 in 13 of 25 windows (expected about 1.25 under a calibrated posterior) | minimum p = 0.000 | median p = 0.040 | KS test of the 25 p-values against U(0,1): p = 0.000
the null is estimated with 100 trials: the smallest attainable p-value is about 0.01


## 4. Power check: deliberately wrong posteriors, same windows, same classifier and null

In [5]:
ctrl = {}
a = so.copy(); a[:, :, 1] += 0.05; ctrl["beta shifted +0.05"] = a
a = so.copy(); a[:, :, 1] += 0.10; ctrl["beta shifted +0.10"] = a
m_ = so.mean(1, keepdims=True); ctrl["width x0.5"] = m_ + 0.5 * (so - m_)
rows = []
for name, s in ctrl.items():
    p = pvals(s); rows.append(dict(control=name, frac_p_below_0_05=float((p < 0.05).mean()), median_p=float(np.median(p)), min_p=float(p.min())))
rows.insert(0, dict(control="production posterior (the test windows above)", frac_p_below_0_05=float((p_main < 0.05).mean()), median_p=float(np.median(p_main)), min_p=float(p_main.min())))
pc = pd.DataFrame(rows); display(pc.round(3)); pc.to_csv(R / "37_N3_lc2st_power_check.csv", index=False)
print("posterior sd of beta in these windows (mean):", round(float(so.std(1)[:, 1].mean()), 3), "| the +0.10 shift is", round(0.10 / float(so.std(1)[:, 1].mean()), 1), "posterior sd")
print(f"notebook wall time {(time.time() - T0) / 60:.1f} min")


,control,frac_p_below_0_05,median_p,min_p
0,production posterior (the test windows above),0.52,0.04,0.0
1,beta shifted +0.05,0.52,0.04,0.0
2,beta shifted +0.10,0.48,0.05,0.0
3,width x0.5,0.52,0.04,0.0


posterior sd of beta in these windows (mean): 0.044 | the +0.10 shift is 2.3 posterior sd
notebook wall time 6.9 min


## 6. Interpretation — to be written after the run
Toward the storyline: what the individual-posterior test says about the production ensemble per window, how much power the default classifier has (the positive control), what that licenses us to say, and what it does not (the NUTS replacement is a check against a
calibration set simulated by the same simulator, not an independent gold standard).